# Instructor Validators — Teaching the LLM to Self-Correct

**Week 1 | Notebook 2 of 4**

**What you'll learn:**
- Why validation fails with raw prompting
- `@field_validator` — custom validation logic
- Instructor retry loop — how it works under the hood
- `max_retries=3` — debugging retry traces
- ValidationError message injection — what the LLM sees
- Complex cross-field validation (domain matches email)
- Tenacity integration — exponential backoff

**Runtime:** ~40 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("01_instructor/02_validators.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  01_instructor/02_validators.ipynb
Task:      Auto-retry with validators
Calls:     ~15

With GPT-4o:       $0.23 USD
With GPT-4o-mini:  $0.02 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup & Why Validation Fails with Raw Prompting

In [2]:
from pydantic import BaseModel, ValidationError, field_validator

from src.config import USE_OLLAMA, get_instructor_client

client = get_instructor_client()  # provider from LLM_PROVIDER in .env (default: openai)

# Raw prompting often produces invalid output
# Example: model returns 'paul.krishai.com' instead of 'paul@krishai.com'
print("❌ Raw prompting has no built-in validation or retry")
print("✅ Instructor catches validation errors and feeds them back to the LLM")

❌ Raw prompting has no built-in validation or retry
✅ Instructor catches validation errors and feeds them back to the LLM


## 2. `@field_validator` — Custom Validation Logic

In [3]:
class EmailAddress(BaseModel):
    address: str
    domain: str
    is_corporate: bool

    @field_validator("address")
    @classmethod
    def must_be_valid_email(cls, v):
        if "@" not in v:
            raise ValueError(f"'{v}' is not a valid email — must contain '@'")
        return v.lower()

    @field_validator("domain")
    @classmethod
    def must_match_address(cls, v, values):
        if "address" in values.data:
            expected = values.data["address"].split("@")[1]
            if v != expected:
                raise ValueError(f"Domain '{v}' doesn't match email domain '{expected}'")
        return v


# Test the validator
try:
    email = EmailAddress(address="paul.krishai.com", domain="krishai.com", is_corporate=True)
except ValidationError as e:
    print("Validation caught the error:")
    print(e)

Validation caught the error:
1 validation error for EmailAddress
address
  Value error, 'paul.krishai.com' is not a valid email — must contain '@' [type=value_error, input_value='paul.krishai.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 3. Instructor Retry Loop — How It Works

In [4]:
# With max_retries — Instructor retries up to 3 times on validation failure
email = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=EmailAddress,
    max_retries=3,  # ← Instructor handles retry + error feedback automatically
    messages=[{"role": "user", "content": "Parse: paul@krishai.com (company email)"}],
)

print(email)

# What happens internally:
# 1. First attempt: model returns valid output → done
# 2. If invalid: Instructor injects error into next prompt
# 3. Model sees error and corrects itself

address='paul@krishai.com' domain='krishai.com' is_corporate=True


## 4. Debugging Retry Traces

In [5]:
# Enable verbose logging to see retries
import logging

logging.basicConfig(level=logging.INFO)


# A model with a strict validator that might fail initially
class StrictURL(BaseModel):
    url: str

    @field_validator("url")
    @classmethod
    def must_be_https(cls, v):
        if not v.startswith("https://"):
            raise ValueError(f"URL must start with https://, got: {v}")
        return v


# This might trigger retries if model returns http://
strict_url = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=StrictURL,
    max_retries=3,
    messages=[{"role": "user", "content": "Extract the secure URL for example.com"}],
)

print(f"Result: {strict_url.url}")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Result: https://example.com


## 5. ValidationError Message Injection — What the LLM Sees

In [6]:
# Simulate what Instructor sends back on failure
error_msg = """ValidationError: 'paul.krishai.com' is not a valid email — must contain '@'.
Please fix and retry."""

print("This is what the LLM sees on retry:")
print(error_msg)

# The model uses this feedback to self-correct
print("\n✅ The LLM learns from its mistake and produces valid output on the next try")

This is what the LLM sees on retry:
ValidationError: 'paul.krishai.com' is not a valid email — must contain '@'.
Please fix and retry.

✅ The LLM learns from its mistake and produces valid output on the next try


## 6. Complex Cross-Field Validation

In [7]:
from datetime import date

from pydantic import model_validator


class Event(BaseModel):
    name: str
    start_date: str
    end_date: str

    @model_validator(mode="after")
    def check_dates(self):
        start = date.fromisoformat(self.start_date)
        end = date.fromisoformat(self.end_date)
        if end < start:
            raise ValueError(f"End date {end} must be after start date {start}")
        return self


event = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=Event,
    max_retries=3,
    messages=[
        {"role": "user", "content": "Conference: AI Summit, starts 2025-06-01, ends 2025-06-03"}
    ],
)

print(event)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


name='AI Summit' start_date='2025-06-01' end_date='2025-06-03'


## 7. Tenacity Integration — Exponential Backoff

In [8]:
from tenacity import retry, stop_after_attempt, wait_exponential


# Wrap Instructor calls with Tenacity for API resilience
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=10))
def resilient_extract(text: str):
    return client.chat.completions.create(
        model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
        response_model=EmailAddress,
        messages=[{"role": "user", "content": text}],
    )


result = resilient_extract("Extract: admin@company.org")
print(result)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


address='admin@company.org' domain='company.org' is_corporate=True
